# ST4 — impact de spromotions , de. la livraison et des retours sur la marge 



L'objectif est de répondre à une question métier simple :

Les promotions et les frais de livraison améliorent-ils réellement la performance commerciale, ou détruisent-ils la marge nette ?

Dans ce notebook, je vais :

1. charger les données propres disponibles dans `data/silver/` ;
2. documenter les hypothèses nécessaires ;
3. calculer la marge brute, les remises, les frais de livraison estimés et la marge nette ;
4. analyser l'impact des promotions, de la livraison, de la ponctualité et de la satisfaction ;
5. produire des fichiers `data/gold/` réutilisables dans le dashboard tableau 
.

## 1. Import des librairies

On utilise des librairies simples :

- `pandas` pour manipuler les données ;
- `numpy` pour les calculs ;
- `plotly` pour les graphiques interactifs ;
- `pathlib` pour gérer les chemins du projet.

In [ ]:
# Manipulation des données
import pandas as pd
import numpy as np

# Visualisation interactive
import plotly.express as px
import plotly.graph_objects as go

# Gestion des chemins
from pathlib import Path

# Affichage plus lisible dans le notebook
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Librairies importées avec succès.")

: 

## 2. Définition des chemins du projet

Le notebook est prévu pour être placé dans le dossier `notebooks/`.


In [ ]:
# Chemin courant
current_path = Path.cwd()

# Si le notebook est lancé depuis le dossier notebooks, on remonte d'un niveau
if current_path.name.lower() == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

# Sécurité : si le dossier data n'est pas trouvé, on essaie le parent
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SILVER_DIR = PROJECT_ROOT / "data" / "silver"
GOLD_DIR = PROJECT_ROOT / "data" / "gold"

# Création du dossier gold si besoin
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print("Racine projet :", PROJECT_ROOT)
print("Dossier Silver :", SILVER_DIR)
print("Dossier Gold :", GOLD_DIR)

## 3. Chargement des données

Pour ST4, on utilise principalement trois fichiers :

| Fichier | Rôle dans l'analyse |
|---|---|
| `train_clean.csv` | données de livraison, remises, coût produit, poids, satisfaction |
| `online_retail_full.csv` | données de ventes e-commerce |
| `online_retail_returns.csv` | données de retours clients |

Le fichier `train_clean.csv` est la base principale pour l'analyse des promotions et de la livraison.

In [ ]:
# Chargement des datasets
shipping = pd.read_csv(SILVER_DIR / "train_clean.csv")
sales = pd.read_csv(SILVER_DIR / "online_retail_full.csv")
returns = pd.read_csv(SILVER_DIR / "online_retail_returns.csv")

# Correction possible du caractère BOM dans le nom de la colonne ID
# Certains CSV peuvent afficher ï»¿ID au lieu de ID.
for df in [shipping, sales, returns]:
    df.columns = (
        df.columns
        .str.replace("\ufeff", "", regex=False)
        .str.replace("ï»¿", "", regex=False)
        .str.strip()
    )

print("Données chargées avec succès.")
print("Shipping :", shipping.shape)
print("Sales :", sales.shape)
print("Returns :", returns.shape)

## 4. Aperçu rapide des données

Avant de commencer les calculs, on vérifie les premières lignes et les colonnes disponibles.

Cette étape permet de s'assurer que les fichiers sont bien chargés et que les noms des colonnes sont corrects.

In [ ]:
print("Colonnes shipping :")
display(pd.DataFrame({"colonnes": shipping.columns}))

print("Aperçu shipping :")
display(shipping.head())

In [ ]:
print("Colonnes sales :")
display(pd.DataFrame({"colonnes": sales.columns}))

print("Aperçu sales :")
display(sales.head())

In [ ]:
print("Colonnes returns :")
display(pd.DataFrame({"colonnes": returns.columns}))

print("Aperçu returns :")
display(returns.head())

## 5. Hypothèses métier importantes

Les deux datasets ne possèdent pas une clé commune directe permettant de relier précisément une ligne de livraison à une facture Online Retail.

Donc, pour ST4, on applique une approche **analytique et documentée** :

1. Les coûts, remises, poids, modes de livraison et satisfaction viennent du dataset `train_clean.csv`.
2. Les ventes et les retours Online Retail servent à calculer un **taux global de retour en valeur**.
3. La perte liée aux retours est ensuite estimée sur chaque commande shipping à partir de ce taux global.
4. Les frais de livraison ne sont pas disponibles dans le dataset, donc on utilise une grille tarifaire hypothétique basée sur le poids et le mode de transport.

Cette hypothèse est nécessaire pour construire une vision de marge complète.

## 6. Préparation des données de ventes et retours

On vérifie que les colonnes nécessaires existent, puis on calcule les indicateurs globaux :

- chiffre d'affaires total ;
- valeur totale des retours ;
- taux global de retour en valeur.

Ce taux sera utilisé ensuite pour estimer une perte retour dans la table ST4.

In [ ]:
# Conversion des dates si elles existent
if "InvoiceDate" in sales.columns:
    sales["InvoiceDate"] = pd.to_datetime(sales["InvoiceDate"], errors="coerce")

if "InvoiceDate" in returns.columns:
    returns["InvoiceDate"] = pd.to_datetime(returns["InvoiceDate"], errors="coerce")

# Calcul du chiffre d'affaires si la colonne n'existe pas
if "TotalRevenue" not in sales.columns:
    sales["TotalRevenue"] = sales["Quantity"] * sales["UnitPrice"]

# Calcul de la valeur des retours si la colonne n'existe pas
if "TotalPrice" not in returns.columns:
    returns["TotalPrice"] = returns["Quantity"] * returns["UnitPrice"]

# Les retours sont souvent négatifs : on prend la valeur absolue pour mesurer la perte.
total_sales_revenue = sales["TotalRevenue"].sum()
total_return_value = returns["TotalPrice"].abs().sum()

# Taux de retour en valeur : montant retourné / chiffre d'affaires total
return_value_rate = total_return_value / total_sales_revenue if total_sales_revenue != 0 else 0

print(f"Chiffre d'affaires total Online Retail : {total_sales_revenue:,.2f} £")
print(f"Valeur totale des retours : {total_return_value:,.2f} £")
print(f"Taux global de retour en valeur : {return_value_rate:.2%}")

### Interprétation

Le taux global de retour en valeur donne une idée de la part du chiffre d'affaires qui est annulée ou perdue à cause des retours.

Dans la suite, ce taux est utilisé comme une hypothèse prudente pour estimer la perte retour dans l'analyse de marge.

## 7. Nettoyage léger du dataset shipping

On prépare maintenant la table principale de ST4.

Les actions effectuées sont simples :

- renommer certaines colonnes pour les rendre plus lisibles ;
- convertir les colonnes numériques ;
- créer des libellés faciles à comprendre pour la ponctualité.

In [ ]:
# Copie de travail pour éviter de modifier le dataframe original
st4 = shipping.copy()

# Renommage de colonnes pour simplifier les manipulations
rename_map = {
    "ID": "order_id",
    "Mode_of_Shipment": "shipment_mode",
    "Discount_offered": "discount_pct",
    "Cost_of_the_Product": "product_cost",
    "Weight_in_gms": "weight_gms",
    "Customer_rating": "customer_rating",
    "Customer_care_calls": "care_calls",
    "Product_importance": "product_importance",
    "Reached.on.Time_Y.N": "on_time"
}

st4 = st4.rename(columns=rename_map)

# Conversion des colonnes numériques importantes
numeric_cols = [
    "discount_pct",
    "product_cost",
    "weight_gms",
    "customer_rating",
    "care_calls",
    "Prior_purchases",
    "on_time"
]

for col in numeric_cols:
    if col in st4.columns:
        st4[col] = pd.to_numeric(st4[col], errors="coerce")

# Libellé lisible de la ponctualité
# Selon le CDC : 1 = Oui, 0 = Non.
st4["delivery_status"] = np.where(st4["on_time"] == 1, "Livré à temps", "Livré en retard")

# Normalisation du texte
st4["shipment_mode"] = st4["shipment_mode"].astype(str).str.strip()
st4["product_importance"] = st4["product_importance"].astype(str).str.strip().str.capitalize()

print("Table ST4 préparée :")
display(st4.head())

## 8. Création des tranches de remise

Le CDC demande d'analyser l'impact promotionnel par niveaux de remise.

On crée donc les tranches suivantes :

| Tranche | Interprétation |
|---|---|
| `0% - Prix plein` | pas de promotion |
| `1-10% - Remise légère` | promotion faible |
| `11-25% - Remise significative` | promotion commerciale importante |
| `26-50% - Promotion forte` | promotion agressive |
| `>50% - Liquidation` | remise très élevée, risque de marge négative |

In [ ]:
# Fonction de segmentation des remises
def build_discount_bucket(discount):
    if pd.isna(discount):
        return "Non renseigné"
    elif discount == 0:
        return "0% - Prix plein"
    elif discount <= 10:
        return "1-10% - Remise légère"
    elif discount <= 25:
        return "11-25% - Remise significative"
    elif discount <= 50:
        return "26-50% - Promotion forte"
    else:
        return ">50% - Liquidation"

st4["discount_bucket"] = st4["discount_pct"].apply(build_discount_bucket)

# Ordre logique pour l'affichage des tranches
bucket_order = [
    "0% - Prix plein",
    "1-10% - Remise légère",
    "11-25% - Remise significative",
    "26-50% - Promotion forte",
    ">50% - Liquidation",
    "Non renseigné"
]

st4["discount_bucket"] = pd.Categorical(
    st4["discount_bucket"],
    categories=bucket_order,
    ordered=True
)

st4["discount_bucket"].value_counts().sort_index()

## 9. Hypothèse de frais de livraison

Les frais de livraison réels ne sont pas disponibles dans le dataset.

On définit donc une hypothèse simple basée sur le poids du colis :

| Mode de livraison | Hypothèse utilisée |
|---|---|
| Ship | `0.002 £ / gramme` |
| Flight | `0.005 £ / gramme` |
| Road | `0.001 £ / gramme` |

Cette hypothèse permet de comparer les modes de transport et d'intégrer la livraison dans la marge nette.

In [ ]:
# Grille tarifaire hypothétique par gramme
shipping_rate_by_mode = {
    "Ship": 0.002,
    "Flight": 0.005,
    "Road": 0.001
}

# Application de la grille tarifaire
st4["shipping_rate"] = st4["shipment_mode"].map(shipping_rate_by_mode).fillna(0.002)
st4["shipping_cost"] = st4["weight_gms"] * st4["shipping_rate"]

# Aperçu du calcul
st4[["shipment_mode", "weight_gms", "shipping_rate", "shipping_cost"]].head()

## 10. Calcul de la marge

On calcule maintenant les composantes principales de la marge :

| Colonne | Signification |
|---|---|
| `estimated_revenue` | revenu estimé avant remise |
| `gross_margin` | marge brute avant remise et livraison |
| `discount_impact` | montant de la remise accordée |
| `shipping_cost` | frais de livraison estimés |
| `return_loss` | perte estimée liée aux retours |
| `net_margin` | marge nette finale |

La formule principale est :

```text
Marge nette = Marge brute - Impact remise - Frais livraison - Perte retour
```

In [ ]:
# Pour éviter une division par zéro, on limite la remise à 95% maximum dans le calcul du revenu estimé.
# Cela ne change pas la colonne discount_pct originale, utilisée pour les analyses.
safe_discount = st4["discount_pct"].clip(lower=0, upper=95)

# Estimation du revenu avant remise à partir du coût produit et du taux de remise.
# Cette logique reprend l'approche du modèle dbt existant du projet.
st4["estimated_revenue"] = st4["product_cost"] / (1 - safe_discount / 100)

# Marge brute estimée
st4["gross_margin"] = st4["estimated_revenue"] - st4["product_cost"]

# Impact financier de la remise
st4["discount_impact"] = st4["estimated_revenue"] * (st4["discount_pct"] / 100)

# Perte retour estimée à partir du taux global observé dans Online Retail
st4["return_loss"] = st4["estimated_revenue"] * return_value_rate

# Marge nette finale
st4["net_margin"] = (
    st4["gross_margin"]
    - st4["discount_impact"]
    - st4["shipping_cost"]
    - st4["return_loss"]
)

# Taux de marge nette
st4["net_margin_rate"] = st4["net_margin"] / st4["estimated_revenue"]

# Arrondi des colonnes financières
money_cols = [
    "estimated_revenue", "gross_margin", "discount_impact",
    "shipping_cost", "return_loss", "net_margin", "net_margin_rate"
]

for col in money_cols:
    st4[col] = st4[col].round(4)

# Aperçu des résultats
st4[[
    "order_id", "shipment_mode", "discount_pct", "product_cost", "estimated_revenue",
    "gross_margin", "discount_impact", "shipping_cost", "return_loss", "net_margin", "net_margin_rate"
]].head()

## 11. KPIs globaux ST4

On résume maintenant la situation générale :

- chiffre d'affaires estimé ;
- marge brute ;
- impact total des remises ;
- frais de livraison estimés ;
- perte retour estimée ;
- marge nette finale.

In [ ]:
kpi_st4 = pd.DataFrame({
    "KPI": [
        "Nombre de commandes",
        "Revenu estimé total",
        "Marge brute totale",
        "Impact total des remises",
        "Frais de livraison estimés",
        "Perte retour estimée",
        "Marge nette totale",
        "Taux moyen de remise",
        "Taux moyen de marge nette"
    ],
    "Valeur": [
        len(st4),
        st4["estimated_revenue"].sum(),
        st4["gross_margin"].sum(),
        st4["discount_impact"].sum(),
        st4["shipping_cost"].sum(),
        st4["return_loss"].sum(),
        st4["net_margin"].sum(),
        st4["discount_pct"].mean(),
        st4["net_margin_rate"].mean()
    ]
})

# Affichage lisible
kpi_st4_display = kpi_st4.copy()
kpi_st4_display["Valeur"] = kpi_st4_display["Valeur"].round(2)
display(kpi_st4_display)

### Lecture métier

Cette table permet de voir rapidement si la marge nette reste positive après l'effet cumulé :

- du coût produit ;
- des remises ;
- de la livraison ;
- des retours.

Si la marge nette est faible ou négative, cela signifie que la politique commerciale doit être ajustée.

## 12. Décomposition de la marge — Waterfall

Le graphique suivant montre la cascade de marge :

```text
Revenu estimé → coût produit → remise → livraison → retour → marge nette
```

C'est un visuel important pour le dashboard ST4, car il explique visuellement où la marge est consommée.

In [ ]:
# Données pour le waterfall
waterfall_data = pd.DataFrame({
    "etape": [
        "Revenu estimé",
        "Coût produit",
        "Remises",
        "Livraison",
        "Retours",
        "Marge nette"
    ],
    "valeur": [
        st4["estimated_revenue"].sum(),
        -st4["product_cost"].sum(),
        -st4["discount_impact"].sum(),
        -st4["shipping_cost"].sum(),
        -st4["return_loss"].sum(),
        st4["net_margin"].sum()
    ],
    "type": ["absolute", "relative", "relative", "relative", "relative", "total"]
})

fig = go.Figure(go.Waterfall(
    name="Marge",
    orientation="v",
    measure=waterfall_data["type"],
    x=waterfall_data["etape"],
    y=waterfall_data["valeur"],
    text=[f"{v:,.0f} £" for v in waterfall_data["valeur"]],
    textposition="outside"
))

fig.update_layout(
    title="Décomposition de la marge nette estimée",
    yaxis_title="Montant estimé (£)",
    showlegend=False
)

fig.show()

## 13. Analyse des remises

On analyse maintenant les performances par tranche de remise.

L'objectif est d'identifier si les remises élevées créent réellement de la valeur ou si elles détruisent la marge.

In [ ]:
promo_analysis = (
    st4
    .groupby("discount_bucket", observed=True)
    .agg(
        nb_commandes=("order_id", "count"),
        remise_moyenne=("discount_pct", "mean"),
        revenu_total=("estimated_revenue", "sum"),
        marge_brute_totale=("gross_margin", "sum"),
        impact_remise_total=("discount_impact", "sum"),
        frais_livraison_total=("shipping_cost", "sum"),
        perte_retour_totale=("return_loss", "sum"),
        marge_nette_totale=("net_margin", "sum"),
        marge_nette_moyenne=("net_margin", "mean"),
        taux_marge_nette_moyen=("net_margin_rate", "mean")
    )
    .reset_index()
)

# Arrondi pour affichage
promo_analysis_display = promo_analysis.copy()
for col in promo_analysis_display.select_dtypes(include=[np.number]).columns:
    promo_analysis_display[col] = promo_analysis_display[col].round(2)

display(promo_analysis_display)

In [ ]:
fig = px.bar(
    promo_analysis,
    x="discount_bucket",
    y="marge_nette_moyenne",
    title="Marge nette moyenne par tranche de remise",
    labels={
        "discount_bucket": "Tranche de remise",
        "marge_nette_moyenne": "Marge nette moyenne (£)"
    },
    text_auto=".2f"
)

fig.update_layout(xaxis_tickangle=-25)
fig.show()

In [ ]:
fig = px.line(
    promo_analysis,
    x="discount_bucket",
    y="taux_marge_nette_moyen",
    markers=True,
    title="Évolution du taux moyen de marge nette selon la remise",
    labels={
        "discount_bucket": "Tranche de remise",
        "taux_marge_nette_moyen": "Taux moyen de marge nette"
    }
)

fig.update_layout(xaxis_tickangle=-25)
fig.show()

### Interprétation métier

Cette partie permet d'identifier la tranche de remise à partir de laquelle la marge nette devient trop faible.

En pratique, si une tranche de remise affiche une marge nette moyenne négative, elle doit être utilisée uniquement pour des opérations particulières : liquidation, déstockage ou acquisition client très ciblée.

## 14. Analyse par mode de livraison

On compare maintenant les modes de livraison :

- `Ship` ;
- `Flight` ;
- `Road`.

L'objectif est d'identifier le meilleur compromis entre coût logistique, ponctualité, satisfaction client et marge nette.

In [ ]:
shipping_mode_analysis = (
    st4
    .groupby("shipment_mode")
    .agg(
        nb_commandes=("order_id", "count"),
        poids_moyen=("weight_gms", "mean"),
        cout_livraison_moyen=("shipping_cost", "mean"),
        taux_livraison_temps=("on_time", "mean"),
        satisfaction_moyenne=("customer_rating", "mean"),
        remise_moyenne=("discount_pct", "mean"),
        marge_nette_moyenne=("net_margin", "mean"),
        taux_marge_nette_moyen=("net_margin_rate", "mean")
    )
    .reset_index()
    .sort_values("marge_nette_moyenne", ascending=False)
)

shipping_mode_display = shipping_mode_analysis.copy()
for col in shipping_mode_display.select_dtypes(include=[np.number]).columns:
    shipping_mode_display[col] = shipping_mode_display[col].round(2)

display(shipping_mode_display)

In [ ]:
fig = px.bar(
    shipping_mode_analysis,
    x="shipment_mode",
    y="cout_livraison_moyen",
    title="Coût moyen de livraison par mode de transport",
    labels={
        "shipment_mode": "Mode de livraison",
        "cout_livraison_moyen": "Coût moyen de livraison (£)"
    },
    text_auto=".2f"
)

fig.show()

In [ ]:
fig = px.bar(
    shipping_mode_analysis,
    x="shipment_mode",
    y="marge_nette_moyenne",
    title="Marge nette moyenne par mode de livraison",
    labels={
        "shipment_mode": "Mode de livraison",
        "marge_nette_moyenne": "Marge nette moyenne (£)"
    },
    text_auto=".2f"
)

fig.show()

In [ ]:
fig = px.scatter(
    shipping_mode_analysis,
    x="cout_livraison_moyen",
    y="marge_nette_moyenne",
    size="nb_commandes",
    color="shipment_mode",
    title="Arbitrage coût livraison vs marge nette par mode",
    labels={
        "cout_livraison_moyen": "Coût moyen de livraison (£)",
        "marge_nette_moyenne": "Marge nette moyenne (£)",
        "shipment_mode": "Mode de livraison"
    }
)

fig.show()

### Interprétation métier

Un bon mode de livraison n'est pas seulement le moins cher.

Il doit aussi préserver :

- la ponctualité ;
- la satisfaction client ;
- la marge nette.

Cette comparaison permet donc d'éviter une décision basée uniquement sur le coût logistique.

## 15. Analyse ponctualité et satisfaction

On étudie ici l'effet de la ponctualité sur la satisfaction et la marge.

L'idée métier est simple : une livraison en retard peut générer plus d'appels SAV, plus d'insatisfaction et potentiellement plus de retours.

In [ ]:
delivery_analysis = (
    st4
    .groupby("delivery_status")
    .agg(
        nb_commandes=("order_id", "count"),
        satisfaction_moyenne=("customer_rating", "mean"),
        appels_sav_moyens=("care_calls", "mean"),
        remise_moyenne=("discount_pct", "mean"),
        cout_livraison_moyen=("shipping_cost", "mean"),
        marge_nette_moyenne=("net_margin", "mean"),
        taux_marge_nette_moyen=("net_margin_rate", "mean")
    )
    .reset_index()
)

delivery_display = delivery_analysis.copy()
for col in delivery_display.select_dtypes(include=[np.number]).columns:
    delivery_display[col] = delivery_display[col].round(2)

display(delivery_display)

In [ ]:
fig = px.bar(
    delivery_analysis,
    x="delivery_status",
    y="satisfaction_moyenne",
    title="Satisfaction moyenne selon la ponctualité de livraison",
    labels={
        "delivery_status": "Statut de livraison",
        "satisfaction_moyenne": "Satisfaction moyenne"
    },
    text_auto=".2f"
)

fig.show()

In [ ]:
fig = px.bar(
    delivery_analysis,
    x="delivery_status",
    y="appels_sav_moyens",
    title="Appels SAV moyens selon la ponctualité de livraison",
    labels={
        "delivery_status": "Statut de livraison",
        "appels_sav_moyens": "Nombre moyen d'appels SAV"
    },
    text_auto=".2f"
)

fig.show()

## 16. Analyse du risque retour

Comme le dataset shipping ne contient pas directement une colonne retour, on construit un **score de risque retour**.

Ce score est une approximation métier basée sur plusieurs signaux :

- remise élevée ;
- livraison en retard ;
- faible satisfaction ;
- nombreux appels au service client ;
- poids élevé du colis.

Ce score ne remplace pas un vrai retour observé, mais il permet de repérer les commandes potentiellement risquées.

In [ ]:
# Score simple et lisible de risque retour
st4["return_risk_score"] = 0

# Une remise forte peut attirer des comportements opportunistes ou réduire la qualité perçue.
st4.loc[st4["discount_pct"] > 25, "return_risk_score"] += 1
st4.loc[st4["discount_pct"] > 50, "return_risk_score"] += 1

# Une livraison en retard augmente le risque d'insatisfaction.
st4.loc[st4["on_time"] == 0, "return_risk_score"] += 1

# Une satisfaction faible est un signal de risque.
st4.loc[st4["customer_rating"] <= 2, "return_risk_score"] += 1

# Beaucoup d'appels SAV indiquent une expérience client plus difficile.
st4.loc[st4["care_calls"] >= st4["care_calls"].quantile(0.75), "return_risk_score"] += 1

# Les colis lourds peuvent coûter plus cher en retour.
st4.loc[st4["weight_gms"] >= st4["weight_gms"].quantile(0.75), "return_risk_score"] += 1

# Catégorie lisible du risque
st4["return_risk_level"] = pd.cut(
    st4["return_risk_score"],
    bins=[-1, 1, 3, 10],
    labels=["Faible", "Moyen", "Élevé"]
)

st4[["discount_pct", "delivery_status", "customer_rating", "care_calls", "weight_gms", "return_risk_score", "return_risk_level"]].head()

In [ ]:
return_risk_analysis = (
    st4
    .groupby("return_risk_level", observed=True)
    .agg(
        nb_commandes=("order_id", "count"),
        remise_moyenne=("discount_pct", "mean"),
        satisfaction_moyenne=("customer_rating", "mean"),
        cout_livraison_moyen=("shipping_cost", "mean"),
        perte_retour_moyenne=("return_loss", "mean"),
        marge_nette_moyenne=("net_margin", "mean"),
        taux_marge_nette_moyen=("net_margin_rate", "mean")
    )
    .reset_index()
)

return_risk_display = return_risk_analysis.copy()
for col in return_risk_display.select_dtypes(include=[np.number]).columns:
    return_risk_display[col] = return_risk_display[col].round(2)

display(return_risk_display)

In [ ]:
fig = px.bar(
    return_risk_analysis,
    x="return_risk_level",
    y="marge_nette_moyenne",
    title="Marge nette moyenne selon le niveau de risque retour",
    labels={
        "return_risk_level": "Niveau de risque retour",
        "marge_nette_moyenne": "Marge nette moyenne (£)"
    },
    text_auto=".2f"
)

fig.show()

## 17. Analyse remise vs marge nette

On visualise maintenant directement la relation entre le taux de remise et la marge nette.

Ce graphique aide à repérer les zones dangereuses : remises élevées avec marge nette faible ou négative.

In [ ]:
fig = px.scatter(
    st4,
    x="discount_pct",
    y="net_margin",
    color="discount_bucket",
    size="estimated_revenue",
    hover_data=["shipment_mode", "customer_rating", "delivery_status", "return_risk_level"],
    title="Relation entre remise accordée et marge nette",
    labels={
        "discount_pct": "Remise accordée (%)",
        "net_margin": "Marge nette estimée (£)",
        "discount_bucket": "Tranche de remise"
    }
)

fig.show()

### Lecture métier

Plus la remise augmente, plus l'entreprise doit vendre en volume pour compenser la perte de marge unitaire.

Si les remises fortes sont associées à une marge nette faible, elles doivent être limitées ou réservées à des cas précis.

## 18. Analyse par importance produit

On regarde maintenant si les produits déclarés comme importants ont un comportement différent :

- coût plus élevé ;
- remise plus forte ;
- meilleure ou moins bonne marge ;
- risque retour plus élevé.

In [ ]:
importance_analysis = (
    st4
    .groupby("product_importance")
    .agg(
        nb_commandes=("order_id", "count"),
        cout_produit_moyen=("product_cost", "mean"),
        remise_moyenne=("discount_pct", "mean"),
        satisfaction_moyenne=("customer_rating", "mean"),
        poids_moyen=("weight_gms", "mean"),
        marge_nette_moyenne=("net_margin", "mean"),
        risque_retour_moyen=("return_risk_score", "mean")
    )
    .reset_index()
    .sort_values("marge_nette_moyenne", ascending=False)
)

importance_display = importance_analysis.copy()
for col in importance_display.select_dtypes(include=[np.number]).columns:
    importance_display[col] = importance_display[col].round(2)

display(importance_display)

In [ ]:
fig = px.bar(
    importance_analysis,
    x="product_importance",
    y="marge_nette_moyenne",
    title="Marge nette moyenne par niveau d'importance produit",
    labels={
        "product_importance": "Importance produit",
        "marge_nette_moyenne": "Marge nette moyenne (£)"
    },
    text_auto=".2f"
)

fig.show()

## 19. Identification des commandes à risque

On extrait maintenant les lignes les plus sensibles :

- marge nette négative ou faible ;
- remise élevée ;
- risque retour élevé.

Cette table est utile dans le dashboard pour montrer les cas prioritaires à surveiller.

In [ ]:
# Commandes à marge négative ou très faible
risk_orders = st4[
    (st4["net_margin"] < 0) |
    (st4["return_risk_level"] == "Élevé") |
    (st4["discount_pct"] > 50)
].copy()

risk_orders = risk_orders.sort_values(
    by=["net_margin", "return_risk_score"],
    ascending=[True, False]
)

risk_orders_display = risk_orders[[
    "order_id", "shipment_mode", "discount_pct", "discount_bucket",
    "product_cost", "estimated_revenue", "shipping_cost", "return_loss",
    "net_margin", "customer_rating", "delivery_status",
    "return_risk_score", "return_risk_level"
]].head(20)

display(risk_orders_display)

## 20. Recommandations métier ST4

À partir des analyses précédentes, on prépare une table de recommandations.

Elle sera exportée dans `data/gold/` et pourra être utilisée dans le rapport final ou dans Power BI.

In [ ]:
recommendations = []

# 1. Seuil de remise recommandé
positive_margin_buckets = promo_analysis[promo_analysis["marge_nette_moyenne"] > 0]
if not positive_margin_buckets.empty:
    last_profitable_bucket = positive_margin_buckets.iloc[-1]["discount_bucket"]
    recommendations.append({
        "theme": "Promotions",
        "constat": f"La dernière tranche encore rentable en moyenne est : {last_profitable_bucket}.",
        "recommandation": "Limiter les remises fortes et suivre systématiquement la marge nette par tranche de remise.",
        "kpi_associe": "marge_nette_moyenne"
    })

# 2. Meilleur mode de livraison selon marge nette moyenne
best_shipping_mode = shipping_mode_analysis.sort_values("marge_nette_moyenne", ascending=False).iloc[0]
recommendations.append({
    "theme": "Livraison",
    "constat": f"Le mode {best_shipping_mode['shipment_mode']} présente la meilleure marge nette moyenne ({best_shipping_mode['marge_nette_moyenne']:.2f} £).",
    "recommandation": "Favoriser ce mode lorsque les contraintes de délai et de qualité le permettent.",
    "kpi_associe": "marge_nette_moyenne"
})

# 3. Risque retour
high_risk_count = st4[st4["return_risk_level"] == "Élevé"].shape[0]
recommendations.append({
    "theme": "Retours",
    "constat": f"{high_risk_count} commandes sont classées avec un risque retour élevé.",
    "recommandation": "Surveiller les commandes combinant forte remise, faible satisfaction, retard et poids élevé.",
    "kpi_associe": "return_risk_score"
})

# 4. Remises très élevées
very_high_discount_count = st4[st4["discount_pct"] > 50].shape[0]
recommendations.append({
    "theme": "Remises fortes",
    "constat": f"{very_high_discount_count} commandes ont une remise supérieure à 50%.",
    "recommandation": "Réserver les remises supérieures à 50% aux opérations de liquidation ou de déstockage contrôlé.",
    "kpi_associe": "discount_pct"
})

recommendations_df = pd.DataFrame(recommendations)
display(recommendations_df)

## 21. Exports pour le dashboard L8

On exporte maintenant les tables utiles dans `data/gold/`.

Ces fichiers peuvent ensuite être importés dans Power BI pour construire la page :

> **ST4 — Marge & Rentabilité**

In [ ]:
# Table principale ST4
st4_export_cols = [
    "order_id", "shipment_mode", "discount_pct", "discount_bucket",
    "product_cost", "weight_gms", "customer_rating", "care_calls",
    "product_importance", "on_time", "delivery_status",
    "estimated_revenue", "gross_margin", "discount_impact",
    "shipping_cost", "return_loss", "net_margin", "net_margin_rate",
    "return_risk_score", "return_risk_level"
]

# On garde uniquement les colonnes présentes pour éviter les erreurs
st4_export_cols = [col for col in st4_export_cols if col in st4.columns]

# Export des fichiers Gold
st4[st4_export_cols].to_csv(GOLD_DIR / "mart_promo_impact_eda.csv", index=False, encoding="utf-8-sig")
promo_analysis.to_csv(GOLD_DIR / "promo_margin_analysis_st4.csv", index=False, encoding="utf-8-sig")
shipping_mode_analysis.to_csv(GOLD_DIR / "shipping_mode_analysis_st4.csv", index=False, encoding="utf-8-sig")
delivery_analysis.to_csv(GOLD_DIR / "delivery_analysis_st4.csv", index=False, encoding="utf-8-sig")
return_risk_analysis.to_csv(GOLD_DIR / "return_risk_analysis_st4.csv", index=False, encoding="utf-8-sig")
importance_analysis.to_csv(GOLD_DIR / "importance_analysis_st4.csv", index=False, encoding="utf-8-sig")
risk_orders[st4_export_cols].to_csv(GOLD_DIR / "risk_orders_st4.csv", index=False, encoding="utf-8-sig")
recommendations_df.to_csv(GOLD_DIR / "recommendations_st4.csv", index=False, encoding="utf-8-sig")
waterfall_data.to_csv(GOLD_DIR / "waterfall_margin_st4.csv", index=False, encoding="utf-8-sig")

print("Exports réalisés avec succès dans :", GOLD_DIR)
print("Fichiers générés :")
for file in [
    "mart_promo_impact_eda.csv",
    "promo_margin_analysis_st4.csv",
    "shipping_mode_analysis_st4.csv",
    "delivery_analysis_st4.csv",
    "return_risk_analysis_st4.csv",
    "importance_analysis_st4.csv",
    "risk_orders_st4.csv",
    "recommendations_st4.csv",
    "waterfall_margin_st4.csv"
]:
    print("-", file)

## 22. Synthèse finale

Ce notebook a permis de construire une analyse complète de la rentabilité e-commerce.

Les principaux résultats produits sont :

1. une table détaillée de marge par commande ;
2. une analyse des remises par tranche ;
3. une comparaison des modes de livraison ;
4. une analyse de la ponctualité et de la satisfaction ;
5. un score de risque retour ;
6. une table de recommandations métier.

Les fichiers exportés dans `data/gold/` sont prêts pour construire la page Power BI :

> **ST4 — Marge & Rentabilité**

Cette page pourra contenir :

- des KPI cards : revenu, marge brute, remise, livraison, retours, marge nette ;
- un waterfall de décomposition de marge ;
- un graphique remise vs marge nette ;
- une comparaison des modes de livraison ;
- une table des commandes à risque ;
- une table de recommandations.